# Hybrid LSTM–Transformer Architecture with Multi-Scale Feature Fusion for High-Accuracy Gold Futures Price Forecasting

**Authors:** Yali Zhao, Yingying Guo, Xuecheng Wang  
**DOI:** [https://doi.org/10.3390/math13101551](https://doi.org/10.3390/math13101551)  

Pedagogical notebook generated by `/wids-make-companion`. Each cell is a self-contained illustration of one section. Run from top to bottom in Colab (CPU runtime is fine).

In [ ]:
!pip install --quiet matplotlib numpy pandas scikit-learn shap statsmodels torch xgboost

## 1. Problem and Motivation: Why Gold Futures Forecasting Is Hard

Gold futures act as a safe-haven asset whose price reacts to macroeconomic policy, geopolitical shocks, and cross-market spillovers, with extreme volatility (e.g., COMEX gold hit USD 3025/oz in March 2025). Traditional models like ARIMA and GARCH assume linearity and static parameters, so they fail to capture nonlinear dynamics, multi-source noise, and abrupt regime shifts. The authors target this gap by building a hybrid deep learning system trained on Shanghai Futures Exchange daily data from 2015 to 2025.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
T = 200

# --- Simulate nonlinear gold price with two regime shifts ---
price = np.zeros(T)
price[0] = 1800.0
for t in range(1, T):
    if t < 70:
        drift = 0.3; vol = 5.0          # calm uptrend
    elif t < 130:
        drift = 2.5; vol = 25.0         # geopolitical shock + high vol
    else:
        drift = -0.5; vol = 8.0         # mean-reversion
    price[t] = price[t - 1] + drift + np.random.normal(0, vol)

train, test = price[:150], price[150:]
fit = ARIMA(train, order=(1, 1, 1)).fit()
forecast = fit.forecast(steps=50)
naive = np.full(50, train[-1])

mae_arima = np.mean(np.abs(forecast - test))
mae_naive = np.mean(np.abs(naive - test))

print("=== Why Linear Models Fail on Regime-Shifting Gold Prices ===")
print(f"Naive (last value) MAE : {mae_naive:.2f}")
print(f"ARIMA(1,1,1)      MAE : {mae_arima:.2f}")
print(f"ARIMA / Naive ratio    : {mae_arima / mae_naive:.2f}x")

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(price, color="goldenrod", label="Synthetic Gold Price")
axes[0].axvline(70, color="red", ls="--", alpha=.6, label="Regime shift 1")
axes[0].axvline(130, color="blue", ls="--", alpha=.6, label="Regime shift 2")
axes[0].axvline(150, color="gray", ls=":", alpha=.8, label="Train/test split")
axes[0].set_title("Simulated Gold Futures with Regime Shifts")
axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

idx = np.arange(150, 200)
axes[1].plot(idx, test, color="goldenrod", label="Actual (test)")
axes[1].plot(idx, forecast, color="crimson", ls="--", label=f"ARIMA MAE={mae_arima:.1f}")
axes[1].plot(idx, naive, color="steelblue", ls=":", label=f"Naive MAE={mae_naive:.1f}")
axes[1].set_title("Forecast Comparison in Post-Shock Regime")
axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 2. Prior Work and Three Identified Research Gaps

A survey of 100+ Chinese and international studies shows progress with ARIMA, GARCH, VAR, BP networks, single LSTM/GRU, CNN-LSTM, XGBoost, and decomposition hybrids (e.g., CEEMDAN-LSTM, VMD-XGBoost), but persistent weaknesses remain. The authors highlight three gaps: (1) shallow serial integration of temporal models with attention, (2) static data partitioning that ignores heterogeneous gold price drivers, and (3) no dynamic adaptation when markets shift regimes during black-swan events.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

np.random.seed(42)

# Regime A (train): calm market
n_train = 200
t_train = np.arange(n_train)
price_A = 1800 + 0.5 * t_train + np.random.normal(0, 5, n_train)

# Regime B (test): black-swan shock, high volatility
n_test = 60
t_test = np.arange(n_train, n_train + n_test)
price_B = 1900 + 2.5 * t_test * 0.1 + np.random.normal(0, 80, n_test)

def make_xy(prices):
    return prices[:-1].reshape(-1, 1), prices[1:]

X_train, y_train = make_xy(price_A)
X_test,  y_test  = make_xy(price_B)

model = LinearRegression().fit(X_train, y_train)
mae_train = mean_absolute_error(y_train, model.predict(X_train))
mae_test  = mean_absolute_error(y_test,  model.predict(X_test))

print("=== Gap 2: Static Partitioning Failure Demo ===")
print(f"Regime A (train) volatility (std): {np.std(np.diff(price_A)):.2f}")
print(f"Regime B (test)  volatility (std): {np.std(np.diff(price_B)):.2f}")
print(f"MAE on Regime A (calm)  : {mae_train:.2f}")
print(f"MAE on Regime B (shock) : {mae_test:.2f}")
print(f"MAE blow-up factor      : {mae_test / mae_train:.1f}x")


## 3. Feature Selection: 25 Indicators Down to 6 via XGBoost + SHAP

From an initial 25 candidate variables across seven dimensions (equity indices, bond yields, FX rates, crypto, alternative-asset ETFs, commodities, and special indicators like VIX and AQI), the authors train an XGBoost model (100 trees, depth 3) and rank features with SHAP values. Six core drivers survive: NASDAQ Composite, S&P 500, silver futures, USD/CNY exchange rate, China 1-year Treasury yield, and the Guotai Zhongzheng Coal ETF. SHAP gives interpretable, signed importance scores so the team can defend each chosen feature on economic grounds.

In [ ]:
import numpy as np
import xgboost as xgb
import shap

np.random.seed(42)
n = 300

nasdaq   = np.random.randn(n)
sp500    = nasdaq * 0.9 + np.random.randn(n) * 0.2
silver   = np.random.randn(n)
usd_cny  = np.random.randn(n)
cn_yield = np.random.randn(n)
coal_etf = np.random.randn(n)

noise = np.random.randn(n, 19)
y = (0.4*nasdaq + 0.3*sp500 + 0.15*silver
     - 0.2*usd_cny + 0.1*cn_yield + 0.25*coal_etf
     + np.random.randn(n) * 0.05)

informative = np.column_stack([nasdaq, sp500, silver, usd_cny, cn_yield, coal_etf])
X = np.hstack([informative, noise])
names = ["NASDAQ", "SP500", "Silver", "USD_CNY", "CN_Yield", "Coal_ETF"] + [f"noise_{i:02d}" for i in range(19)]

model = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                         random_state=42, verbosity=0).fit(X, y)
shap_values = shap.TreeExplainer(model).shap_values(X)
mean_abs = np.abs(shap_values).mean(axis=0)
order = np.argsort(mean_abs)[::-1]

print("Top-6 features by mean |SHAP|:")
for rank, idx in enumerate(order[:6], 1):
    print(f"  {rank}. {names[idx]:<10}  mean|SHAP| = {mean_abs[idx]:.4f}")


## 4. Hybrid LSTM-Transformer Architecture with Cross-Attention

The model combines a 128-neuron bidirectional LSTM (for long-term temporal memory via forget/input/output gates) with a 4-head self-attention Transformer (64-dim keys, with QKV projections and positional encoding) to capture global dependencies. The novelty is bidirectional cross-attention: Transformer outputs feed the LSTM and LSTM hidden states feed back into the Transformer's attention weights, instead of the usual stacked or parallel design. Residual connections and layer normalization keep gradients stable across the dual-path encoder-decoder.

In [ ]:
import torch
import torch.nn as nn

SEQ_LEN = 10
BATCH   = 2
HIDDEN  = 16
D_MODEL = 32
NHEADS  = 2

lstm = nn.LSTM(D_MODEL, HIDDEN, batch_first=True, bidirectional=True)
mha  = nn.MultiheadAttention(D_MODEL, NHEADS, batch_first=True)
cross_lstm_q = nn.MultiheadAttention(D_MODEL, NHEADS, batch_first=True)
cross_xfmr_q = nn.MultiheadAttention(D_MODEL, NHEADS, batch_first=True)
norm1, norm2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)

torch.manual_seed(42)
x = torch.randn(BATCH, SEQ_LEN, D_MODEL)

lstm_out, _ = lstm(x)
print("BiLSTM output:        ", lstm_out.shape)

xfmr_out, _ = mha(x, x, x)
print("Transformer SA output:", xfmr_out.shape)

# LSTM hidden as Q, Transformer as K,V
cross_a, _ = cross_lstm_q(query=lstm_out, key=xfmr_out, value=xfmr_out)
cross_a = norm1(cross_a + lstm_out)

# Transformer as Q, LSTM hidden as K,V
cross_b, _ = cross_xfmr_q(query=xfmr_out, key=lstm_out, value=lstm_out)
cross_b = norm2(cross_b + xfmr_out)

fused = cross_a + cross_b
print("Fused output:         ", fused.shape)


## 5. DHPF and Dual-Loop Adaptive Mechanism

The Dynamic Hierarchical Partition Framework (DHPF) splits data along four axes (price trend via mean filtering, volatility via GARCH, external correlations via Granger causality, event intensity via sentiment) so the model is trained and stress-tested on regime-aware partitions; the train/test split is 70/30 anchored at December 2022 and includes Fed hikes, the Russia-Ukraine conflict, and the SVB collapse in the test set. A dual-loop adaptive mechanism then keeps the model honest at inference time: an outer loop integrates new data every 90 days with Kalman filtering, and an inner loop fires EWMA-based incremental updates when a 3-sigma volatility threshold is crossed, all under a volatility-adjusted loss that adds GARCH(1,1) variance and TVaR(0.95) terms.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 200
low_vol  = np.random.normal(0, 0.01, n // 2)
high_vol = np.random.normal(0, 0.05, n // 2)
returns  = np.concatenate([low_vol, high_vol])
idx = pd.date_range("2022-01-01", periods=n, freq="B")
df = pd.DataFrame({"ret": returns}, index=idx)

df["ewma_mean"] = df["ret"].ewm(span=20).mean()
df["ewma_std"]  = df["ret"].ewm(span=20).std()
df["z"]         = (df["ret"] - df["ewma_mean"]).abs() / df["ewma_std"].replace(0, np.nan)
df["trigger"]   = df["z"] > 3

trigger_idx = np.where(df["trigger"])[0]
print(f"Trigger fires      : {df['trigger'].sum()}")
print(f"In low-vol  half   : {(trigger_idx < n//2).sum()}")
print(f"In high-vol half   : {(trigger_idx >= n//2).sum()}")


## 6. Empirical Results vs. Baselines

On the test set (2022-2024 daily data, 783 samples), the LSTM-Transformer scores R-squared = 0.9618, MAPE = 3.18%, MAE = 421.01, and RMSE = 528.26, beating standalone LSTM (R^2 0.8430), Transformer (0.7763), PatchTST, CNN-LSTM, TCN-Informer, and CNN-GRU-Attention. The paper claims roughly 47% MAE reduction vs. LSTM and 60% vs. Transformer, plus a 62% lower extreme-event prediction error, framing the win as multi-scale fusion (LSTM short-term dynamics + Transformer global context) rather than any single mechanism.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = ["LSTM", "Transformer", "CNN-LSTM", "PatchTST", "LSTM-Transformer\n(Proposed)"]
mae    = [795.0, 1052.0, 610.0, 580.0, 421.01]
r2     = [0.843, 0.776, 0.860, 0.850, 0.9618]

x = np.arange(len(models))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("LSTM-Transformer vs. Baselines (Test 2022-2024, n=783)")

colors = ["#FF9800", "#f44336", "#FF9800", "#FF9800", "#4CAF50"]
ax1.bar(x, mae, color=colors); ax1.set_xticks(x); ax1.set_xticklabels(models, fontsize=9)
ax1.set_ylabel("MAE (lower is better)"); ax1.set_title("Mean Absolute Error")

ax2.bar(x, r2,  color=colors); ax2.set_xticks(x); ax2.set_xticklabels(models, fontsize=9)
ax2.set_ylabel("R^2 (higher is better)"); ax2.set_title("R-Squared"); ax2.set_ylim(0.6, 1.02)

plt.tight_layout(); plt.show()
print("LSTM-Transformer: R^2=0.9618, ~47% MAE reduction vs LSTM, ~60% vs Transformer.")


## 7. Limitations and Future Work

The hybrid model's parameter count and computational cost are noticeably higher than single-architecture models (training time ~86s vs. ~32-47s for baselines), and the gating-plus-global-attention combination reacts more slowly to high-frequency local patterns, so locally sensitive models like CNN-GRU-Attention beat it on raw MAE for short bursts. The authors propose sparse attention to cut redundancy, dynamic regularization, and multi-frequency feature fusion, and suggest extending the framework with high-frequency factors like implied volatility and cross-market contagion models.

In [ ]:
import numpy as np

seq_lengths = [16, 32, 64, 128]
k = 8
d = 64

print(f"{'N':>6} {'Dense FLOPs':>14} {'Sparse FLOPs':>14} {'Ratio':>8} {'Savings':>10}")
print("-" * 60)
for N in seq_lengths:
    dense = 2 * N * N * d
    sparse = 2 * N * k * d
    ratio = dense / sparse
    savings = (1 - sparse / dense) * 100
    print(f"{N:>6} {dense:>14,} {sparse:>14,} {ratio:>7.1f}x {savings:>9.1f}%")
print()
print("Sparse attention: O(N*k) instead of O(N^2). For N=128, k=8 -> 16x cheaper.")
